# 3D reporter timelapse — 04b_reporter_background_stability_qc

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Reporter Background Stability QC

This notebook reviews reporter-background behavior before reporter quantification.

The goal is to identify positions with unstable whole off-cyst background, flag likely late-frame failures, and write reusable QC tables that notebook 05 can consume without recomputing this step.


## Setup

This section loads the post-mask QC table, acquisition metadata, and helper functions needed to measure whole off-cyst reporter background over time.


In [ ]:
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile as tiff
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
import sys

cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    format_display_hours_from_index as _format_display_hours_from_index,
    offset_time_hours_df as _offset_time_hours_df,
    set_display_time_axis as _set_display_time_axis,
)

DATASET_DIR = ROOT / "data/raw/20260128_BMP4-reporter_LPM-organoids/d2-d5"
MASK_METRICS_PATH = ROOT / "results/ilastik/qc/full_dataset_v1_mask_metrics.tsv"
ACQUISITION_SUMMARY_PATH = ROOT / "results/qc/acquisition_qc_summary.json"
ILLUMINATION_FIELD_PATH = ROOT / "results/qc/04_masked_illumination_fields.npz"

FIGURE_DIR = ROOT / "results/figures/04b"
TABLE_DIR = ROOT / "results/tables"
EXECUTED_NOTEBOOK_DIR = ROOT / "results/executed_notebooks"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

mask_metrics = pd.read_csv(MASK_METRICS_PATH, sep="\t", low_memory=False)
acquisition_summary = json.loads(ACQUISITION_SUMMARY_PATH.read_text())
interval_minutes = float(acquisition_summary["interval_minutes"])
mask_metrics["time_hours"] = mask_metrics["time_index"].astype(float) * interval_minutes / 60.0

payload = np.load(ILLUMINATION_FIELD_PATH)
illumination_fields = {
    "RFP": payload["rfp_field"].astype(np.float32),
    "YFP": payload["yfp_field"].astype(np.float32),
}

POSITION_RE = re.compile(r"Pos(?P<position_index>\d+)$")
REPORTER_COLORS = {"RFP": "tab:red", "YFP": "tab:green"}
ISSUE_COLORS = {
    "": "0.75",
    "gradual drift": "goldenrod",
    "broad drift": "tab:orange",
    "sudden jump": "crimson",
}
mask_path_lookup = {
    (str(row.position_label), int(row.time_index)): Path(row.mask_path)
    for row in mask_metrics.itertuples(index=False)
}

print("Project root:", ROOT)
print("Mask metric rows:", len(mask_metrics))
print("Interval minutes:", interval_minutes)


In [ ]:
def position_index_from_label(position_label: str) -> int:
    match = POSITION_RE.fullmatch(position_label)
    if not match:
        raise ValueError(f"Unexpected position label: {position_label}")
    return int(match.group("position_index"))


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours_from_index(time_index: int | float, decimals: int = 1) -> str:
    return _format_display_hours_from_index(
        time_index,
        interval_hours=interval_minutes / 60.0,
        decimals=decimals,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    return _offset_time_hours_df(
        df,
        should_offset_column=lambda column: "time_hours" in str(column).lower(),
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )


def raw_frame_path(position_label: str, channel_index: int, time_index: int) -> Path:
    position_index = position_index_from_label(position_label)
    return (
        DATASET_DIR
        / position_label
        / f"img_channel{channel_index:03d}_position{position_index:03d}_time{time_index:09d}_z000.tif"
    )


def load_reporter(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    channel_index = 1 if reporter == "RFP" else 2
    return tiff.imread(raw_frame_path(position_label, channel_index, time_index))


def load_phase(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(raw_frame_path(position_label, 0, time_index))


def load_reporter_after_illumination(position_label: str, reporter: str, time_index: int) -> np.ndarray:
    image = load_reporter(position_label, reporter, time_index).astype(np.float32)
    return image / np.clip(illumination_fields[reporter], 1e-6, None)


def load_mask(position_label: str, time_index: int) -> np.ndarray:
    return tiff.imread(mask_path_lookup[(position_label, int(time_index))]).astype(bool)


def whole_background_statistics(
    signal: np.ndarray,
    organoid_mask: np.ndarray,
) -> dict[str, object]:
    values = signal[~organoid_mask].astype(float) if np.any(~organoid_mask) else np.array([], dtype=float)
    if values.size:
        background_value = float(np.median(values))
        q25, q75 = np.quantile(values, [0.25, 0.75])
        mad = float(np.median(np.abs(values - background_value)))
    else:
        background_value = float("nan")
        q25 = float("nan")
        q75 = float("nan")
        mad = float("nan")
    return {
        "values": values,
        "background_value": background_value,
        "iqr": float(q75 - q25) if np.isfinite(q25) and np.isfinite(q75) else float("nan"),
        "mad": mad,
    }


def robust_image_limits(image_list: list[np.ndarray], q_low: float, q_high: float) -> tuple[float, float]:
    values = []
    for image in image_list:
        array = np.asarray(image, dtype=float)
        finite = array[np.isfinite(array)]
        if finite.size:
            values.append(finite.reshape(-1))
    if not values:
        return 0.0, 1.0
    pooled = np.concatenate(values)
    vmin = float(np.quantile(pooled, q_low))
    vmax = float(np.quantile(pooled, q_high))
    if not np.isfinite(vmin):
        vmin = float(np.nanmin(pooled))
    if not np.isfinite(vmax):
        vmax = float(np.nanmax(pooled))
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmax = vmin + 1.0
    return vmin, vmax


def choose_review_time_indices(
    available_time_indices: list[int],
    flagged_start_time_index: int,
    total_post_frames: int = 4,
    preferred_pre_gap: int = 16,
) -> list[int]:
    if not available_time_indices:
        return []
    ordered = sorted({int(t) for t in available_time_indices})
    earliest = ordered[0]
    latest = ordered[-1]
    flagged_start = int(np.clip(flagged_start_time_index, earliest, latest))

    pre_candidates = [t for t in ordered if t <= flagged_start - preferred_pre_gap]
    if pre_candidates:
        pre_time = pre_candidates[-1]
    else:
        pre_candidates = [t for t in ordered if t < flagged_start]
        pre_time = pre_candidates[-1] if pre_candidates else ordered[0]

    post_candidates = [t for t in ordered if t >= flagged_start]
    if not post_candidates:
        post_candidates = [ordered[-1]]
    sample_points = np.linspace(0, len(post_candidates) - 1, total_post_frames)
    post_times = [post_candidates[int(round(idx))] for idx in sample_points]
    selected = []
    for time_index in [pre_time] + post_times:
        if time_index not in selected:
            selected.append(int(time_index))
    return selected


## Measure Whole Off-Cyst Background Stability

This section measures whole off-cyst background on the illumination-corrected reporter images for every retained frame, then compares each position trace against robust aggregate background statistics over time.


In [ ]:
background_summary_rows = []
background_trace_rows = []

for reporter in REPORTER_COLORS:
    subset = mask_metrics.loc[
        ~mask_metrics["exclude_from_analysis"]
    ].sort_values(["position_label", "time_index"])
    for position_label, group in subset.groupby("position_label", sort=True):
        time_indices = group["time_index"].astype(int).tolist()
        time_hours = group["time_hours"].astype(float).tolist()
        background_values = []
        for time_index in time_indices:
            mask = load_mask(position_label, time_index)
            signal = load_reporter_after_illumination(position_label, reporter, time_index).astype(float)
            whole_info = whole_background_statistics(
                signal=signal,
                organoid_mask=mask,
            )
            background_values.append(float(whole_info["background_value"]))

        for time_index, time_hour, background_value in zip(time_indices, time_hours, background_values):
            background_trace_rows.append(
                {
                    "position_label": position_label,
                    "reporter": reporter,
                    "time_index": int(time_index),
                    "time_hours": float(time_hour),
                    "background_value": float(background_value),
                }
            )

        background_values = np.asarray(background_values, dtype=float)
        diff_signed = np.diff(background_values)
        diff_abs = np.abs(diff_signed) if background_values.size > 1 else np.array([np.nan], dtype=float)
        max_step_idx = int(np.nanargmax(diff_abs)) if np.isfinite(diff_abs).any() else -1
        max_step_prev = int(time_indices[max_step_idx]) if max_step_idx >= 0 else -1
        max_step_curr = int(time_indices[max_step_idx + 1]) if max_step_idx >= 0 and (max_step_idx + 1) < len(time_indices) else -1
        background_range = float(np.nanmax(background_values) - np.nanmin(background_values)) if background_values.size else float("nan")
        net_change = float(background_values[-1] - background_values[0]) if background_values.size else float("nan")
        background_diff_median = float(np.nanmedian(diff_abs))
        background_diff_max = float(np.nanmax(diff_abs))
        background_diff_ratio = (
            float(background_diff_max / max(background_diff_median, 1e-6))
            if np.isfinite(background_diff_max) and np.isfinite(background_diff_median)
            else float("nan")
        )
        max_step_frac_of_range = (
            float(background_diff_max / background_range)
            if np.isfinite(background_range) and background_range > 0
            else float("nan")
        )
        net_change_frac_of_range = (
            float(abs(net_change) / background_range)
            if np.isfinite(background_range) and background_range > 0
            else float("nan")
        )
        if np.isfinite(net_change) and abs(net_change) > 0 and diff_signed.size > 0:
            finite_signed = diff_signed[np.isfinite(diff_signed)]
            if finite_signed.size:
                sign_consistency = float(np.mean(np.sign(finite_signed) == np.sign(net_change)))
            else:
                sign_consistency = float("nan")
        else:
            sign_consistency = float("nan")

        background_summary_rows.append(
            {
                "position_label": position_label,
                "reporter": reporter,
                "frame_count": int(group.shape[0]),
                "background_median": float(np.nanmedian(background_values)),
                "background_range": background_range,
                "background_diff_median": background_diff_median,
                "background_diff_p95": float(np.nanquantile(diff_abs, 0.95)),
                "background_diff_max": background_diff_max,
                "background_diff_ratio": background_diff_ratio,
                "background_net_change": net_change,
                "net_change_frac_of_range": net_change_frac_of_range,
                "max_step_frac_of_range": max_step_frac_of_range,
                "sign_consistency": sign_consistency,
                "first_time_index": int(time_indices[0]) if time_indices else -1,
                "last_time_index": int(time_indices[-1]) if time_indices else -1,
                "max_step_time_prev": max_step_prev,
                "max_step_time_curr": max_step_curr,
            }
        )

background_trace_df = pd.DataFrame(background_trace_rows)
aggregate_background_df = (
    background_trace_df.groupby(["reporter", "time_index"], as_index=False)
    .agg(
        time_hours=("time_hours", "median"),
        aggregate_background_median=("background_value", "median"),
        aggregate_background_mad=("background_value", lambda s: float(np.median(np.abs(s - np.median(s))))),
    )
)
aggregate_background_df["aggregate_background_sigma"] = 1.4826 * aggregate_background_df["aggregate_background_mad"]
aggregate_background_df["aggregate_background_sigma"] = np.where(
    aggregate_background_df["reporter"] == "RFP",
    np.clip(aggregate_background_df["aggregate_background_sigma"], 50.0, None),
    np.clip(aggregate_background_df["aggregate_background_sigma"], 100.0, None),
)
aggregate_background_df = aggregate_background_df.sort_values(["reporter", "time_index"]).reset_index(drop=True)
for reporter in ["RFP", "YFP"]:
    reporter_mask = aggregate_background_df["reporter"] == reporter
    for column in [
        "aggregate_background_median",
        "aggregate_background_sigma",
    ]:
        aggregate_background_df.loc[reporter_mask, f"{column}_smooth"] = (
            aggregate_background_df.loc[reporter_mask, column]
            .rolling(7, center=True, min_periods=4)
            .median()
            .to_numpy()
        )
aggregate_background_df["aggregate_background_minus_3sigma"] = (
    aggregate_background_df["aggregate_background_median_smooth"]
    - 3.0 * aggregate_background_df["aggregate_background_sigma_smooth"]
)
aggregate_background_df["aggregate_background_plus_3sigma"] = (
    aggregate_background_df["aggregate_background_median_smooth"]
    + 3.0 * aggregate_background_df["aggregate_background_sigma_smooth"]
)

def longest_true_run(mask: np.ndarray) -> tuple[int, int, int]:
    arr = np.asarray(mask, dtype=bool)
    best_len = 0
    best_start = -1
    best_end = -1
    run_start = None
    for idx, is_true in enumerate(arr):
        if is_true and run_start is None:
            run_start = idx
        elif (not is_true) and run_start is not None:
            run_len = idx - run_start
            if run_len > best_len:
                best_len = run_len
                best_start = run_start
                best_end = idx - 1
            run_start = None
    if run_start is not None:
        run_len = len(arr) - run_start
        if run_len > best_len:
            best_len = run_len
            best_start = run_start
            best_end = len(arr) - 1
    return best_len, best_start, best_end

MAIN_ALIGNED_Z_THRESHOLD = 4.0
MAIN_ALIGNED_Z_RUN = 8
RFP_LATE_LOW_TAIL_Z_THRESHOLD = -2.25
RFP_LATE_LOW_TAIL_RUN = 8
RFP_LATE_LOW_TAIL_LAST_FRAMES = 35

deviation_trace_rows = []
background_qc_rows = []
for summary_row in background_summary_rows:
    position_label = str(summary_row["position_label"])
    reporter = str(summary_row["reporter"])
    trace_df = background_trace_df.loc[
        (background_trace_df["position_label"] == position_label)
        & (background_trace_df["reporter"] == reporter)
    ].sort_values("time_index")
    aggregate_df = aggregate_background_df.loc[
        aggregate_background_df["reporter"] == reporter
    ]
    merged = trace_df.merge(
        aggregate_df,
        on=["reporter", "time_index", "time_hours"],
        how="left",
    ).sort_values("time_index")
    merged["background_value_smooth"] = (
        merged["background_value"].rolling(7, center=True, min_periods=4).median()
    )
    merged["signed_deviation"] = (
        merged["background_value_smooth"] - merged["aggregate_background_median_smooth"]
    )
    early_mask = merged["time_index"] < 40
    early_values = merged.loc[early_mask, "signed_deviation"].to_numpy(dtype=float)
    early_values = early_values[np.isfinite(early_values)]
    early_offset = float(np.median(early_values)) if early_values.size else 0.0
    merged["aligned_signed_deviation"] = merged["signed_deviation"] - early_offset
    merged["aligned_robust_z"] = merged["aligned_signed_deviation"] / np.clip(
        merged["aggregate_background_sigma_smooth"],
        1e-6,
        None,
    )
    merged["abs_aligned_robust_z"] = np.abs(merged["aligned_robust_z"])
    merged["outside_3sigma"] = merged["abs_aligned_robust_z"] >= 3.0
    merged["outside_4sigma"] = merged["abs_aligned_robust_z"] >= MAIN_ALIGNED_Z_THRESHOLD

    outside_3sigma_mask = merged["outside_3sigma"].fillna(False).to_numpy(dtype=bool)
    outside_4sigma_mask = merged["outside_4sigma"].fillna(False).to_numpy(dtype=bool)
    longest_4sigma_run, run4_start, run4_end = longest_true_run(outside_4sigma_mask)
    late_tail_time_index_min = int(
        max(
            int(merged["time_index"].min()),
            int(merged["time_index"].max()) - (RFP_LATE_LOW_TAIL_LAST_FRAMES - 1),
        )
    )
    merged["rfp_late_low_tail_candidate"] = (
        (reporter == "RFP")
        & (merged["time_index"] >= late_tail_time_index_min)
        & (merged["aligned_robust_z"] <= RFP_LATE_LOW_TAIL_Z_THRESHOLD)
    )
    late_low_tail_mask = merged["rfp_late_low_tail_candidate"].fillna(False).to_numpy(dtype=bool)
    longest_late_low_tail_run, late_tail_start, late_tail_end = longest_true_run(late_low_tail_mask)
    outside_3sigma_fraction = float(np.nanmean(outside_3sigma_mask.astype(float))) if len(outside_3sigma_mask) else 0.0
    outside_4sigma_fraction = float(np.nanmean(outside_4sigma_mask.astype(float))) if len(outside_4sigma_mask) else 0.0
    max_abs_aligned_z = float(np.nanmax(merged["abs_aligned_robust_z"])) if np.isfinite(merged["abs_aligned_robust_z"]).any() else float("nan")

    flag_candidates = []
    if longest_4sigma_run >= MAIN_ALIGNED_Z_RUN and run4_start >= 0:
        flag_candidates.append(
            {
                "rule": f"aligned_{int(MAIN_ALIGNED_Z_THRESHOLD)}sigma_{MAIN_ALIGNED_Z_RUN}frames",
                "first_idx": int(run4_start),
                "last_idx": int(run4_end),
                "priority": 0,
            }
        )
    if (
        reporter == "RFP"
        and longest_late_low_tail_run >= RFP_LATE_LOW_TAIL_RUN
        and late_tail_start >= 0
    ):
        flag_candidates.append(
            {
                "rule": (
                    "rfp_late_low_tail_"
                    f"{str(abs(RFP_LATE_LOW_TAIL_Z_THRESHOLD)).replace('.', 'p')}sigma_"
                    f"{RFP_LATE_LOW_TAIL_RUN}frames_last{RFP_LATE_LOW_TAIL_LAST_FRAMES}"
                ),
                "first_idx": int(late_tail_start),
                "last_idx": int(late_tail_end),
                "priority": 1,
            }
        )

    flagged = bool(flag_candidates)

    if flag_candidates:
        selected_candidate = sorted(
            flag_candidates,
            key=lambda row: (
                row["priority"],
                row["first_idx"],
                -(row["last_idx"] - row["first_idx"]),
            ),
        )[0]
        first_flag_idx = int(selected_candidate["first_idx"])
        last_flag_idx = int(selected_candidate["last_idx"])
        flag_rule = "; ".join(row["rule"] for row in flag_candidates)
        primary_flag_rule = str(selected_candidate["rule"])
    else:
        first_flag_idx = -1
        last_flag_idx = -1
        flag_rule = ""
        primary_flag_rule = ""

    if flagged and first_flag_idx >= 0:
        flagged_window = merged.iloc[first_flag_idx : last_flag_idx + 1]
        signed_values = flagged_window["aligned_signed_deviation"].to_numpy(dtype=float)
        signed_values = signed_values[np.isfinite(signed_values)]
        median_signed = float(np.median(signed_values)) if signed_values.size else 0.0
        if primary_flag_rule.startswith("rfp_late_low_tail"):
            issue_class = "late low aligned tail"
            issue_label = "late RFP background tail-off"
            issue_detail = (
                "After early-offset alignment, the RFP background stays unusually low near the end of the movie."
            )
        elif median_signed > 0:
            issue_class = "high aligned deviation"
            issue_label = "background rose above the aggregate trend"
            issue_detail = (
                "After early-offset alignment, the background stays above the dataset-wide aggregate trend for a sustained window."
            )
        elif median_signed < 0:
            issue_class = "low aligned deviation"
            issue_label = "background fell below the aggregate trend"
            issue_detail = (
                "After early-offset alignment, the background stays below the dataset-wide aggregate trend for a sustained window."
            )
        else:
            issue_class = "mixed aligned deviation"
            issue_label = "background diverged from the aggregate trend"
            issue_detail = (
                "After early-offset alignment, the background departs from the dataset-wide aggregate trend in a sustained way."
            )
    else:
        issue_class = ""
        issue_label = ""
        issue_detail = ""

    first_flag_time_index = int(merged.iloc[first_flag_idx]["time_index"]) if first_flag_idx >= 0 else -1
    last_flag_time_index = int(merged.iloc[last_flag_idx]["time_index"]) if last_flag_idx >= 0 else -1
    first_flag_time_hours = float(merged.iloc[first_flag_idx]["time_hours"]) if first_flag_idx >= 0 else float("nan")
    last_flag_time_hours = float(merged.iloc[last_flag_idx]["time_hours"]) if last_flag_idx >= 0 else float("nan")

    severity_score = max(
        float(longest_4sigma_run / 8.0),
        float(max_abs_aligned_z / 4.0) if np.isfinite(max_abs_aligned_z) else 0.0,
    )

    deviation_trace_rows.extend(merged.to_dict("records"))
    background_qc_rows.append(
        {
            **summary_row,
            "early_offset": early_offset,
            "outside_3sigma_fraction": outside_3sigma_fraction,
            "outside_4sigma_fraction": outside_4sigma_fraction,
            "longest_4sigma_run": int(longest_4sigma_run),
            "longest_late_low_tail_run": int(longest_late_low_tail_run),
            "max_abs_aligned_z": max_abs_aligned_z,
            "issue_class": issue_class,
            "issue_label": issue_label,
            "issue_detail": issue_detail,
            "flag_rule": flag_rule,
            "primary_flag_rule": primary_flag_rule,
            "flagged": flagged,
            "severity_score": float(severity_score if flagged else 0.0),
            "first_flag_time_index": int(first_flag_time_index),
            "first_flag_time_hours": float(first_flag_time_hours),
            "last_flag_time_index": int(last_flag_time_index),
            "last_flag_time_hours": float(last_flag_time_hours),
        }
    )

background_qc_df = pd.DataFrame(background_qc_rows)
deviation_trace_df = pd.DataFrame(deviation_trace_rows)

flagged_background_df = background_qc_df.loc[background_qc_df["flagged"]].copy()
flagged_background_df = flagged_background_df.sort_values(
    ["severity_score", "reporter", "position_label"],
    ascending=[False, True, True],
).reset_index(drop=True)
if flagged_background_df.empty:
    flagged_position_windows = pd.DataFrame(
        columns=[
            "position_label",
            "position_flagged_reporters",
            "position_first_flag_time_index",
            "position_last_flag_time_index",
            "position_first_flag_time_hours",
            "position_last_flag_time_hours",
        ]
    )
    flagged_position_qc = pd.DataFrame()
    background_exclusion_windows = pd.DataFrame(
        columns=[
            "position_label",
            "background_qc_cut_start_time_index",
            "background_qc_cut_start_time_hours",
            "background_qc_cut_end_time_index",
            "background_qc_cut_end_time_hours",
            "background_qc_cut_to_end",
            "background_qc_flagged_reporters",
            "background_qc_primary_rules",
        ]
    )
else:
    flagged_position_windows = (
        flagged_background_df.groupby("position_label", as_index=False)
        .agg(
            position_flagged_reporters=("reporter", lambda s: ", ".join(sorted(set(map(str, s))))),
            position_first_flag_time_index=("first_flag_time_index", "min"),
            position_last_flag_time_index=("last_flag_time_index", "max"),
            position_first_flag_time_hours=("first_flag_time_hours", "min"),
            position_last_flag_time_hours=("last_flag_time_hours", "max"),
        )
    )
    position_time_extents = (
        mask_metrics.groupby("position_label", as_index=False)
        .agg(
            background_qc_cut_end_time_index=("time_index", "max"),
            background_qc_cut_end_time_hours=("time_hours", "max"),
        )
    )
    background_exclusion_windows = (
        flagged_background_df.groupby("position_label", as_index=False)
        .agg(
            background_qc_cut_start_time_index=("first_flag_time_index", "min"),
            background_qc_cut_start_time_hours=("first_flag_time_hours", "min"),
            background_qc_flagged_reporters=("reporter", lambda s: ", ".join(sorted(set(map(str, s))))),
            background_qc_primary_rules=("primary_flag_rule", lambda s: "; ".join(sorted(set(filter(None, map(str, s)))))),
        )
        .merge(position_time_extents, on="position_label", how="left")
    )
    background_exclusion_windows["background_qc_cut_to_end"] = True
    background_exclusion_windows = background_exclusion_windows[
        [
            "position_label",
            "background_qc_cut_start_time_index",
            "background_qc_cut_start_time_hours",
            "background_qc_cut_end_time_index",
            "background_qc_cut_end_time_hours",
            "background_qc_cut_to_end",
            "background_qc_flagged_reporters",
            "background_qc_primary_rules",
        ]
    ].sort_values("position_label").reset_index(drop=True)
    flagged_background_df = flagged_background_df.merge(
        flagged_position_windows,
        on="position_label",
        how="left",
    )
    background_qc_df = background_qc_df.merge(
        flagged_position_windows,
        on="position_label",
        how="left",
    )
    flagged_position_qc = (
        flagged_background_df.sort_values("severity_score", ascending=False)
        .drop_duplicates("position_label")
        .reset_index(drop=True)
    )
flagged_background_positions = flagged_position_qc["position_label"].tolist()

background_trace_table_path = TABLE_DIR / "04b_background_trace_by_frame.tsv"
background_outlier_table_path = TABLE_DIR / "04b_background_stability_outliers.tsv"
aggregate_background_table_path = TABLE_DIR / "04b_background_aggregate_stats_by_frame.tsv"
background_exclusion_window_path = TABLE_DIR / "04b_background_exclusion_windows.tsv"
background_trace_df.to_csv(background_trace_table_path, sep="\t", index=False)
flagged_background_df.to_csv(background_outlier_table_path, sep="\t", index=False)
aggregate_background_df.to_csv(aggregate_background_table_path, sep="\t", index=False)
background_exclusion_windows.to_csv(background_exclusion_window_path, sep="\t", index=False)

display(
    Markdown(
        f'''
        **Whole off-cyst background stability QC**

        - flagged reporter-position traces: `{len(flagged_background_df)}`
        - flagged positions (union across reporters): `{len(flagged_background_positions)}`
        - black curve below = aggregate median background trace
        - gray band below = aggregate median ± `3 x robust sigma`
        - colored flagged traces are those whose **early-offset-aligned** background residual stays outside the aggregate median by:
          - `4 sigma` for `8` frames
          - or, for `RFP`, a sustained late low tail below `-2.25 sigma` for `8` frames in the last `35` frames
        - `early-offset-aligned` means each position is first shifted to match the aggregate trace over its early baseline window, so we are flagging **drift away from the shared trend**, not just a constant offset

        Wrote trace table: `{background_trace_table_path.name}`  
        Wrote outlier table: `{background_outlier_table_path.name}`  
        Wrote aggregate trace stats: `{aggregate_background_table_path.name}`  
        Wrote exclusion windows: `{background_exclusion_window_path.name}`
        '''
    )
)

if flagged_background_df.empty:
    display(Markdown("No whole-background outliers were flagged by the current rules."))
else:
    display(
        display_time_df(background_exclusion_windows)
    )
    display(
        display_time_df(flagged_background_df[
            [
                "position_label",
                "reporter",
                "issue_label",
                "issue_detail",
                "background_range",
                "background_diff_median",
                "background_diff_max",
                "background_diff_ratio",
                "background_net_change",
                "flag_rule",
                "primary_flag_rule",
                "outside_4sigma_fraction",
                "longest_4sigma_run",
                "longest_late_low_tail_run",
                "max_abs_aligned_z",
                "severity_score",
                "first_flag_time_index",
                "last_flag_time_index",
                "position_first_flag_time_index",
                "position_last_flag_time_index",
            ]
        ])
    )


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15.5, 10.8), sharex=True, constrained_layout=True)
legend_handles = [
    Line2D([0], [0], color="0.75", linewidth=1.2, alpha=0.45, label="All other positions"),
    Line2D([0], [0], color="black", linewidth=1.6, label="Aggregate median"),
    Line2D([0], [0], color="lightgray", linewidth=6.0, alpha=0.35, label="Aggregate median ± 3 sigma"),
    Line2D([0], [0], color="tab:blue", linewidth=2.2, label="Flagged for review (colored by position)"),
]

for axis_index, reporter in enumerate(["RFP", "YFP"]):
    ax = axes[axis_index]
    trace_subset = background_trace_df.loc[background_trace_df["reporter"] == reporter].copy()
    aggregate_subset = aggregate_background_df.loc[
        aggregate_background_df["reporter"] == reporter
    ].sort_values("time_hours")
    summary_subset = background_qc_df.loc[background_qc_df["reporter"] == reporter].copy()
    summary_by_position = summary_subset.set_index("position_label")
    flagged_position_labels = (
        summary_subset.loc[summary_subset["flagged"], "position_label"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    cmap = plt.get_cmap("tab20")
    flagged_color_map = {
        position_label: cmap(color_index % 20)
        for color_index, position_label in enumerate(flagged_position_labels)
    }
    flagged_points = []

    ax.fill_between(
        aggregate_subset["time_hours"],
        aggregate_subset["aggregate_background_minus_3sigma"],
        aggregate_subset["aggregate_background_plus_3sigma"],
        color="lightgray",
        alpha=0.35,
        linewidth=0,
        zorder=0,
    )
    ax.plot(
        aggregate_subset["time_hours"],
        aggregate_subset["aggregate_background_median_smooth"],
        color="black",
        linewidth=1.6,
        alpha=0.9,
        zorder=2,
    )

    for position_label, group in trace_subset.groupby("position_label", sort=True):
        group = group.sort_values("time_hours").reset_index(drop=True)
        summary_row = summary_by_position.loc[position_label]
        flagged = bool(summary_row["flagged"])
        color = flagged_color_map.get(position_label, "0.75") if flagged else "0.75"
        linewidth = 2.0 if flagged else 0.9
        alpha = 0.9 if flagged else 0.28
        zorder = 3 if flagged else 1
        ax.plot(
            group["time_hours"],
            group["background_value"],
            color=color,
            linewidth=linewidth,
            alpha=alpha,
            zorder=zorder,
        )
        if flagged and not group.empty:
            flag_time = int(summary_row["position_first_flag_time_index"])
            flag_hit = group.loc[group["time_index"] == flag_time]
            label_row = flag_hit.iloc[0] if not flag_hit.empty else group.iloc[-1]
            flagged_points.append(
                {
                    "position_label": position_label,
                    "time_hours": float(label_row["time_hours"]),
                    "background_value": float(label_row["background_value"]),
                    "color": color,
                }
            )

    flagged_points = sorted(flagged_points, key=lambda row: row["background_value"])
    y_offsets = [-20, -10, 0, 10, 20, 30, -30, 40, -40, 50, -50, 60]
    for label_index, row in enumerate(flagged_points):
        offset = y_offsets[label_index % len(y_offsets)]
        ax.scatter(
            [row["time_hours"]],
            [row["background_value"]],
            color=row["color"],
            s=18,
            zorder=4,
        )
        ax.annotate(
            row["position_label"],
            (row["time_hours"], row["background_value"]),
            xytext=(4, offset),
            textcoords="offset points",
            fontsize=8.5,
            color=row["color"],
            ha="left",
            va="center",
            zorder=5,
            clip_on=False,
        )

    ax.set_title(f"{reporter} whole off-cyst background over time vs aggregate median +/- 3 sigma")
    ax.set_ylabel("Whole off-cyst background")
    ax.margins(x=0.08)
    ax.grid(alpha=0.2)
    if axis_index == 0:
        ax.legend(handles=legend_handles, loc="upper left", fontsize=8.5, ncol=4, frameon=False)

axes[-1].set_xlabel("Time (hours)")
set_display_time_axis(axes[-1], "x")

background_trace_fig_path = FIGURE_DIR / "04b_background_stability_traces.png"
fig.savefig(background_trace_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote background stability trace summary:", background_trace_fig_path)


## Review Flagged Background Traces

This section shows each flagged position directly against the aggregate median and sigma band, then follows with representative images for the same position.

For each flagged position:
- the first figure is the individual `RFP` / `YFP` background trace review
- the second figure is a representative image montage with all three channels
- image timepoints include at least one frame before the flagged window and several frames after the flagged window starts


In [ ]:
flagged_positions_for_review = (
    flagged_background_df.sort_values("severity_score", ascending=False)
    .drop_duplicates("position_label")["position_label"]
    .tolist()
)

if not flagged_positions_for_review:
    display(Markdown("No background traces were flagged for further review by the current aggregate-range rule."))
else:
    display(
        display_time_df(flagged_background_df[
            [
                "position_label",
                "reporter",
                "issue_label",
                "issue_detail",
                "flag_rule",
                "primary_flag_rule",
                "outside_4sigma_fraction",
                "longest_4sigma_run",
                "longest_late_low_tail_run",
                "max_abs_aligned_z",
                "first_flag_time_index",
                "last_flag_time_index",
                "position_first_flag_time_index",
                "position_last_flag_time_index",
                "severity_score",
            ]
        ].sort_values(["severity_score", "position_label", "reporter"], ascending=[False, True, True]))
    )

    for position_label in flagged_positions_for_review:
        position_flags = flagged_background_df.loc[
            flagged_background_df["position_label"] == position_label
        ].sort_values("severity_score", ascending=False)
        row_title_parts = [
            f"{row.reporter} {row.issue_label}"
            for row in position_flags.itertuples(index=False)
        ]

        union_start_time_index = int(position_flags.iloc[0]["position_first_flag_time_index"])
        union_end_time_index = int(position_flags.iloc[0]["position_last_flag_time_index"])
        union_start_time = float(position_flags.iloc[0]["position_first_flag_time_hours"])
        union_end_time = float(position_flags.iloc[0]["position_last_flag_time_hours"])

        trace_fig, trace_axes = plt.subplots(
            1,
            2,
            figsize=(15.5, 4.6),
            sharex=True,
            constrained_layout=True,
        )
        trace_fig.suptitle(
            (
                f"{position_label} background trace review | "
                f"union t={format_display_hours(union_start_time, 0)}-{format_display_hours(union_end_time, 0)} | "
                + ", ".join(row_title_parts)
            ),
            fontsize=12,
            y=1.02,
        )

        trace_subset_all = background_trace_df.loc[
            background_trace_df["position_label"] == position_label
        ].sort_values("time_index")
        review_time_indices = choose_review_time_indices(
            available_time_indices=trace_subset_all["time_index"].astype(int).tolist(),
            flagged_start_time_index=union_start_time_index,
            total_post_frames=4,
            preferred_pre_gap=16,
        )
        review_time_lookup = (
            trace_subset_all.loc[
                trace_subset_all["time_index"].isin(review_time_indices),
                ["time_index", "time_hours"],
            ]
            .drop_duplicates("time_index")
            .set_index("time_index")["time_hours"]
            .to_dict()
        )

        for col_index, reporter in enumerate(["RFP", "YFP"]):
            ax = trace_axes[col_index]
            aggregate_subset = aggregate_background_df.loc[
                aggregate_background_df["reporter"] == reporter
            ].sort_values("time_hours")
            trace_df = background_trace_df.loc[
                (background_trace_df["position_label"] == position_label)
                & (background_trace_df["reporter"] == reporter)
            ].sort_values("time_hours")
            flag_row = position_flags.loc[position_flags["reporter"] == reporter]

            ax.fill_between(
                aggregate_subset["time_hours"],
                aggregate_subset["aggregate_background_minus_3sigma"],
                aggregate_subset["aggregate_background_plus_3sigma"],
                color="lightgray",
                alpha=0.35,
                linewidth=0,
                zorder=0,
            )
            ax.plot(
                aggregate_subset["time_hours"],
                aggregate_subset["aggregate_background_median_smooth"],
                color="black",
                linewidth=1.4,
                alpha=0.9,
                zorder=1,
            )
            ax.plot(
                trace_df["time_hours"],
                trace_df["background_value"],
                color=REPORTER_COLORS[reporter],
                linewidth=2.0 if not flag_row.empty else 1.3,
                alpha=0.95 if not flag_row.empty else 0.65,
                zorder=2,
            )

            if np.isfinite(union_start_time) and np.isfinite(union_end_time) and union_end_time >= union_start_time:
                ax.axvspan(union_start_time, union_end_time, color="0.4", alpha=0.08, linewidth=0, zorder=0.5)
            if np.isfinite(union_start_time):
                ax.axvline(union_start_time, color="0.25", linestyle="--", linewidth=1.0, zorder=3)
            if not flag_row.empty:
                start_time = float(flag_row.iloc[0]["first_flag_time_hours"])
                end_time = float(flag_row.iloc[0]["last_flag_time_hours"])
                if np.isfinite(start_time):
                    ax.axvline(start_time, color=REPORTER_COLORS[reporter], linestyle=":", linewidth=1.1)
                if np.isfinite(end_time):
                    ax.axvline(end_time, color=REPORTER_COLORS[reporter], linestyle=":", linewidth=1.1, alpha=0.8)
            for marker_time_index in review_time_indices:
                marker_time_hours = review_time_lookup.get(int(marker_time_index))
                if marker_time_hours is not None and np.isfinite(marker_time_hours):
                    ax.axvline(marker_time_hours, color="0.5", linestyle=":", linewidth=0.8, alpha=0.45, zorder=0.9)

            ax.set_title(reporter)
            ax.grid(alpha=0.2)
            if col_index == 0:
                ax.set_ylabel("Whole off-cyst background")
            ax.set_xlabel("Time (hours)")
            set_display_time_axis(ax, "x")

        trace_fig_path = FIGURE_DIR / f"04b_background_review_{position_label}_traces.png"
        trace_fig.savefig(trace_fig_path, dpi=180, bbox_inches="tight")
        display(trace_fig)
        plt.close(trace_fig)
        print("Wrote flagged background trace review:", trace_fig_path)

        phase_images = [load_phase(position_label, time_index).astype(np.float32) for time_index in review_time_indices]
        rfp_images = [load_reporter_after_illumination(position_label, "RFP", time_index).astype(np.float32) for time_index in review_time_indices]
        yfp_images = [load_reporter_after_illumination(position_label, "YFP", time_index).astype(np.float32) for time_index in review_time_indices]
        masks = [load_mask(position_label, time_index) for time_index in review_time_indices]

        phase_vmin, phase_vmax = robust_image_limits(phase_images, 0.01, 0.995)
        rfp_vmin, rfp_vmax = robust_image_limits(rfp_images, 0.005, 0.995)
        yfp_vmin, yfp_vmax = robust_image_limits(yfp_images, 0.005, 0.995)

        image_fig, image_axes = plt.subplots(
            3,
            len(review_time_indices),
            figsize=(2.6 * len(review_time_indices), 8.0),
            constrained_layout=True,
        )
        if len(review_time_indices) == 1:
            image_axes = np.asarray(image_axes).reshape(3, 1)

        channel_specs = [
            ("Phase", phase_images, "gray", phase_vmin, phase_vmax),
            ("RFP (illum-corrected)", rfp_images, "magma", rfp_vmin, rfp_vmax),
            ("YFP (illum-corrected)", yfp_images, "magma", yfp_vmin, yfp_vmax),
        ]
        pre_time_index = review_time_indices[0] if review_time_indices else None
        for row_index, (row_label, image_list, cmap_name, vmin, vmax) in enumerate(channel_specs):
            for col_index, time_index in enumerate(review_time_indices):
                ax = image_axes[row_index, col_index]
                ax.imshow(image_list[col_index], cmap=cmap_name, vmin=vmin, vmax=vmax)
                ax.contour(masks[col_index].astype(float), levels=[0.5], colors="cyan", linewidths=0.8)
                sample_label = "pre-window" if (pre_time_index is not None and time_index == pre_time_index and col_index == 0) else "post-start"
                ax.set_title(
                    f"t={format_display_hours(float(review_time_lookup[int(time_index)]), 0)}\n{sample_label}",
                    fontsize=9,
                )
                ax.set_xticks([])
                ax.set_yticks([])
                if col_index == 0:
                    ax.set_ylabel(row_label, fontsize=10)

        image_fig.suptitle(
            f"{position_label} representative images around flagged window",
            fontsize=12,
            y=1.02,
        )
        image_fig_path = FIGURE_DIR / f"04b_background_review_{position_label}_images.png"
        image_fig.savefig(image_fig_path, dpi=180, bbox_inches="tight")
        display(image_fig)
        plt.close(image_fig)
        print("Wrote flagged background image review:", image_fig_path)


## Outputs

This notebook writes reusable background-QC tables for notebook 05:

- `results/tables/04b_background_trace_by_frame.tsv`
- `results/tables/04b_background_stability_outliers.tsv`
- `results/tables/04b_background_aggregate_stats_by_frame.tsv`
- `results/tables/04b_background_exclusion_windows.tsv`
- `results/figures/04b/04b_background_stability_traces.png`
- `results/figures/04b/04b_background_review_Pos*_traces.png`
- `results/figures/04b/04b_background_review_Pos*_images.png`
